# Exp-A/B/C/DXY/Spot Feature Comparison Experiment

This notebook compares several feature engineering ideas against the baseline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.append("..")

from src.processing import load_and_clean_data
from src.features import generate_features, frac_diff
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, summarize_ic

# Load data
df_raw = load_and_clean_data("../data/BOJ_data.xlsx", "../data/BOJ_meeting_history.csv")
print(f"Data loaded. Shape: {df_raw.shape}")

Data loaded. Shape: (4653, 30)


## Baseline (42 features)

In [2]:
# 1. Baseline
df_feat_base = generate_features(df_raw)  # Default: all flags False
df_pooled_base = pool_boj_data(df_feat_base)

res_3d_base = walk_forward_validation(df_pooled_base, "Target_3d_norm", "2024-01-01")
res_5d_base = walk_forward_validation(df_pooled_base, "Target_5d_norm", "2024-01-01")

ic_3d_base = summarize_ic(res_3d_base)
ic_5d_base = summarize_ic(res_5d_base)

print("Baseline IC (3d):", round(ic_3d_base["ic_all"], 4))
print("Baseline IC (5d):", round(ic_5d_base["ic_all"], 4))
# Expected: 3d OOS IC approx 0.224, 5d approx 0.228

Baseline IC (3d): 0.224
Baseline IC (5d): 0.2277


## Exp-A: Weekday Cyclic Encoding

In [3]:
# Exp-A
df_feat_a = generate_features(df_raw, add_weekday_cyclic=True)
df_pooled_a = pool_boj_data(df_feat_a)

res_3d_a = walk_forward_validation(df_pooled_a, "Target_3d_norm", "2024-01-01")
res_5d_a = walk_forward_validation(df_pooled_a, "Target_5d_norm", "2024-01-01")

ic_3d_a = summarize_ic(res_3d_a)
ic_5d_a = summarize_ic(res_5d_a)

## Exp-B: Days_to_MPM Cyclic Encoding

In [4]:
# Exp-B
df_feat_b = generate_features(df_raw, add_days_to_mpm_cyclic=True)
df_pooled_b = pool_boj_data(df_feat_b)

res_3d_b = walk_forward_validation(df_pooled_b, "Target_3d_norm", "2024-01-01")
res_5d_b = walk_forward_validation(df_pooled_b, "Target_5d_norm", "2024-01-01")

ic_3d_b = summarize_ic(res_3d_b)
ic_5d_b = summarize_ic(res_5d_b)

## Exp-C: Curve Features

In [5]:
# Exp-C
df_feat_c = generate_features(df_raw, add_curve_features=True)
df_pooled_c = pool_boj_data(df_feat_c)

res_3d_c = walk_forward_validation(df_pooled_c, "Target_3d_norm", "2024-01-01")
res_5d_c = walk_forward_validation(df_pooled_c, "Target_5d_norm", "2024-01-01")

ic_3d_c = summarize_ic(res_3d_c)
ic_5d_c = summarize_ic(res_5d_c)

## Exp-DXY: Add DXY_frac_diff

In [6]:
def generate_features_with_dxy(df, d=0.4, window=50):
    feat_df = generate_features(df, d=d, window=window)
    if "DXY" in feat_df.columns:
        feat_df["DXY_frac_diff"] = frac_diff(feat_df["DXY"], d=d, window=window)
    return feat_df

df_feat_dxy = generate_features_with_dxy(df_raw)
df_pooled_dxy = pool_boj_data(df_feat_dxy)

# Manually add DXY_frac_diff if missing from pooled columns
if "DXY_frac_diff" not in df_pooled_dxy.columns:
    print("DXY_frac_diff missing from pooled, merging manually...")
    dxy_map = df_feat_dxy[["Date", "DXY_frac_diff"]].drop_duplicates()
    df_pooled_dxy = df_pooled_dxy.merge(dxy_map, on="Date", how="left")

res_3d_dxy = walk_forward_validation(df_pooled_dxy, "Target_3d_norm", "2024-01-01")
res_5d_dxy = walk_forward_validation(df_pooled_dxy, "Target_5d_norm", "2024-01-01")

ic_3d_dxy = summarize_ic(res_3d_dxy)
ic_5d_dxy = summarize_ic(res_5d_dxy)

DXY_frac_diff missing from pooled, merging manually...


## Exp-Spot: Remove T12/T18/T24

In [7]:
def generate_features_no_spot(df, d=0.4, window=50):
    df_no_spot = df.drop(columns=[c for c in ["T12", "T18", "T24"] if c in df.columns])
    return generate_features(df_no_spot, d=d, window=window)

df_feat_no_spot = generate_features_no_spot(df_raw)
df_pooled_no_spot = pool_boj_data(df_feat_no_spot)

res_3d_no_spot = walk_forward_validation(df_pooled_no_spot, "Target_3d_norm", "2024-01-01")
res_5d_no_spot = walk_forward_validation(df_pooled_no_spot, "Target_5d_norm", "2024-01-01")

ic_3d_no_spot = summarize_ic(res_3d_no_spot)
ic_5d_no_spot = summarize_ic(res_5d_no_spot)

## Summary of Results

In [8]:
results = {
    "Baseline": {
        "3d_ic_all": ic_3d_base["ic_all"],
        "3d_ic_recent": ic_3d_base["ic_recent"],
        "5d_ic_all": ic_5d_base["ic_all"],
        "5d_ic_recent": ic_5d_base["ic_recent"],
    },
    "Exp-A (+weekday)": {
        "3d_ic_all": ic_3d_a["ic_all"],
        "3d_ic_recent": ic_3d_a["ic_recent"],
        "5d_ic_all": ic_5d_a["ic_all"],
        "5d_ic_recent": ic_5d_a["ic_recent"],
    },
    "Exp-B (+Days_to_MPM)": {
        "3d_ic_all": ic_3d_b["ic_all"],
        "3d_ic_recent": ic_3d_b["ic_recent"],
        "5d_ic_all": ic_5d_b["ic_all"],
        "5d_ic_recent": ic_5d_b["ic_recent"],
    },
    "Exp-C (+curve)": {
        "3d_ic_all": ic_3d_c["ic_all"],
        "3d_ic_recent": ic_3d_c["ic_recent"],
        "5d_ic_all": ic_5d_c["ic_all"],
        "5d_ic_recent": ic_5d_c["ic_recent"],
    },
    "Exp-DXY (+DXY_fd)": {
        "3d_ic_all": ic_3d_dxy["ic_all"],
        "3d_ic_recent": ic_3d_dxy["ic_recent"],
        "5d_ic_all": ic_5d_dxy["ic_all"],
        "5d_ic_recent": ic_5d_dxy["ic_recent"],
    },
    "Exp-Spot (no T12/18/24)": {
        "3d_ic_all": ic_3d_no_spot["ic_all"],
        "3d_ic_recent": ic_3d_no_spot["ic_recent"],
        "5d_ic_all": ic_5d_no_spot["ic_all"],
        "5d_ic_recent": ic_5d_no_spot["ic_recent"],
    },
}

df_results = pd.DataFrame(results).T
df_results["3d_delta"] = df_results["3d_ic_all"] - df_results.loc["Baseline", "3d_ic_all"]
df_results["5d_delta"] = df_results["5d_ic_all"] - df_results.loc["Baseline", "5d_ic_all"]
print(df_results.round(4).to_string())

                         3d_ic_all  3d_ic_recent  5d_ic_all  5d_ic_recent  3d_delta  5d_delta
Baseline                    0.2240        0.4140     0.2277        0.4494    0.0000    0.0000
Exp-A (+weekday)            0.2465        0.4234     0.2252        0.4128    0.0225   -0.0025
Exp-B (+Days_to_MPM)        0.2453        0.4440     0.2037        0.3965    0.0213   -0.0240
Exp-C (+curve)              0.2000        0.3470     0.1700        0.3660   -0.0240   -0.0577
Exp-DXY (+DXY_fd)           0.2240        0.4140     0.2277        0.4494    0.0000    0.0000
Exp-Spot (no T12/18/24)     0.1835        0.4004     0.2306        0.4604   -0.0405    0.0029
